In [16]:
import pandas as pd
import ast #Para convertir strings en listas
import re
import json
import os

Lista de funciones que se usaran 

In [12]:
# Funciones

#------------------------------------------------------------------------------------------
# Convierte las funciones de YOLO a coordenadas que puedo tomar para el calculo del area
#------------------------------------------------------------------------------------------

def yolo_to_xyxy(box):
    # box = [class, cx, cy, w, h]
    _, cx, cy, w, h = box
    xmin = cx - w/2
    ymin = cy - h/2
    xmax = cx + w/2
    ymax = cy + h/2
    return xmin, ymin, xmax, ymax

#------------------------------------------------------------------------------------------
# A partir de las coordenadas entregadas por la funcion yolo_to_xyxy se construyen cajas
# que se usaran para el calculo de la IoU
#------------------------------------------------------------------------------------------

def iou(box1, box2):
    x1_min, y1_min, x1_max, y1_max = yolo_to_xyxy(box1)
    x2_min, y2_min, x2_max, y2_max = yolo_to_xyxy(box2)

    # intersección
    inter_xmin = max(x1_min, x2_min)
    inter_ymin = max(y1_min, y2_min)
    inter_xmax = min(x1_max, x2_max)
    inter_ymax = min(y1_max, y2_max)

    inter_area = max(0, inter_xmax - inter_xmin) * max(0, inter_ymax - inter_ymin)

    # áreas individuales
    area1 = (x1_max - x1_min) * (y1_max - y1_min)
    area2 = (x2_max - x2_min) * (y2_max - y2_min)

    union = area1 + area2 - inter_area
    return inter_area / union if union > 0 else 0.0

#------------------------------------------------------------------------------------------
# Para los casos en los que existan multiples gt y pred boxes lo mejor es construir una
# matriz que contenga todos los valores para su futuro procesamiento.
#------------------------------------------------------------------------------------------

def iou_matrix_antigua(gt_boxes, pred_boxes):
    """
    gt_boxes: lista de cajas en formato YOLO [cls, cx, cy, w, h]
    pred_boxes: lista de cajas en formato YOLO [cls, cx, cy, w, h]
    """
    matriz = []
    for i, gt in enumerate(gt_boxes):
        fila = []
        for j, pred in enumerate(pred_boxes):
            fila.append(round(iou(gt, pred), 2))  # redondeamos a 2 decimales
        matriz.append(fila)

    # etiquetas tipo GT₀, P₀...
    df = pd.DataFrame(matriz, 
                      index=[f"GT{i}" for i in range(len(gt_boxes))],
                      columns=[f"P{j}" for j in range(len(pred_boxes))])
    return df

def iou_matrix(gt_boxes, pred_boxes):
    """Construye matriz IoU de todos los GT vs todos los Pred."""
    if not gt_boxes or not pred_boxes:
        return []
    matrix = []
    for gt in gt_boxes:
        row = []
        for pred in pred_boxes:
            row.append(round(iou(gt, pred), 4))  # redondeo opcional
        matrix.append(row)
    return matrix

#------------------------------------------------------------------------------------------
# Es necesario convertir columnas de texto a columnas de floats para facilitar su procesa
# miento
#------------------------------------------------------------------------------------------

def parse_segment_column(cell):
    """Convierte texto tipo "['1 0.8 0.5 0.3 0.3', ...]" a lista de listas de floats."""
    if pd.isna(cell):
        return []
    try:
        return [list(map(float, s.strip().split())) for s in ast.literal_eval(cell)]
    except Exception:
        return []

In [23]:
# ---------- Procesamiento principal ----------
file_path = "..\\..\\Resultados_Segmentacion\\YoloV11_Segmentation-Results_i7-1185G7_Batch_1_parallelV2_4_workers.csv"
df = pd.read_csv(file_path)

# Todas las columnas de predicciones que terminan en "- segment"
pred_cols = [col for col in df.columns if re.search(r"- segment$", col)]
gt_col = "Segments"

for pred_col in pred_cols:
    # Nombre de nueva columna
    iou_col_name = f"IoU_matrix_{pred_col.split(' ')[0]}"
    iou_matrices = []

    for _, row in df.iterrows():
        gt_boxes = parse_segment_column(row[gt_col])
        pred_boxes = parse_segment_column(row[pred_col])
        matrix = iou_matrix(gt_boxes, pred_boxes)
        iou_matrices.append(json.dumps(matrix))  # guardamos como string JSON

    df[iou_col_name] = iou_matrices

In [24]:
# Guardar el archivo

base_name = os.path.basename(file_path)              # "YoloV11_Clasification-Results_i7-1185G7_Batch_1_parallel_4_workers.csv"
name_no_ext, ext = os.path.splitext(base_name)      # ("YoloV11_Clasification-Results_i7-1185G7_Batch_1_parallel_4_workers", ".csv")

out_name = f"{name_no_ext}_with_IoU{ext}"

df.to_csv(out_name, index=False)

print(f"Archivo guardado como: {out_name}")

Archivo guardado como: YoloV11_Segmentation-Results_i7-1185G7_Batch_1_parallelV2_4_workers_with_IoU.csv


# Ejemplo de la forma en la que se usan las funciones

In [13]:
# ------------------- EJEMPLO -------------------
gt = [
    [1.0, 0.802344, 0.532407, 0.284896, 0.937037], # GT0
    [1.0, 0.5,      0.5,      0.3,      0.3     ]  # GT1
]

pred = [
    [1.0, 0.815023, 0.522201, 0.263590, 0.908162], # P0
    [1.0, 0.803424, 0.670529, 0.224537, 0.621631], # P1
    [1.0, 0.4,      0.4,      0.25,     0.25    ]  # P2
]

df = iou_matrix(gt, pred)
print(df)

df = iou_matrix_antigua(gt, pred)
print(df)

[[0.8837, 0.5229, 0.0], [0.0, 0.0, 0.2513]]
       P0    P1    P2
GT0  0.88  0.52  0.00
GT1  0.00  0.00  0.25
